# Scheduling: decisions, storage and command order

Schedule a small `6×8 @ 8×6` matrix multiplication. Change storage capacity, reuse loaded blocks, and inspect the actual commands and checks.

Open this notebook with the [shared launcher](README.md#open-the-notebooks), then choose **Kernel → Restart Kernel and Run All Cells**. Widgets are optional; the ordinary Python calls work too. No board or target compiler is needed.

The dimensions, capacities and instruction declarations below are teaching inputs. See the [scheduling guide](../../docs/guides/05-transformations.md) for the real backend boundary.


**Step 3 of 5: Scheduling** | [All tutorials](README.md) | [Previous: Test generation](02_test_generation.ipynb) | [Next: Transformations](04_transformations.ipynb)


In [ ]:
from collections import Counter
from dataclasses import replace
from html import escape

import numpy as np
from xdsl.dialects.builtin import ArrayAttr, IntegerAttr, ModuleOp, StringAttr, i64
from xdsl.ir import Block, Region
from xdsl.utils.exceptions import VerifyException

from merlin.compiler.schedule.blocks import (
    BANK_ALIGNED,
    CONTIGUOUS,
    OPPOSITE_END,
    BlockScheduleError,
    Contraction,
    Geometry,
    Knobs,
    Load,
    Store,
    check_residency,
    execute,
    schedule_contraction,
)
from merlin.compiler.schedule.schedule_ir import (
    dense_schedule_text,
    parse_dense_schedule_plan,
)
from merlin.sched.check.static import check_kernel
from merlin.sched.ir import Kernel, Ptr, TensorArg, Var, call, loop, parse_kernel
from merlin.sched.isa import InstrDef, InstructionSet, Operand
from merlin.xdsl_dialects.schedule import BindOp, HandleType, LoopTransformOp, PlanOp

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None


def show_html(markup, text):
    if HTML is None:
        print(text)
    else:
        display(HTML(markup))


## 1. Record a compiler decision

First, look at the actual xDSL `schedule` dialect. xDSL is the Python compiler framework used here. `schedule.bind` names a program fragment, and `schedule.loop_transform` requests a loop split.

The cell prints real IR, parses it again and checks its structure. A split factor of one is rejected. This teaching binding names no real payload: parsing the request does not execute it. The implemented dense route in `compiler.schedule.schedule_ir.read_dense_schedule_plan` separately checks the original source/context and supported decisions.


In [ ]:
def split_decision(factor=2):
    bound = BindOp.create(
        properties={"target": StringAttr("illustrative.copy_bytes")},
        result_types=[HandleType()],
    )
    split = LoopTransformOp.create(
        operands=[bound.handle],
        properties={
            "action": StringAttr("split"),
            "path": ArrayAttr([StringAttr("i")]),
            "factor": IntegerAttr(factor, i64),
        },
    )
    plan = PlanOp.create(
        properties={"sym_name": StringAttr("split_demo")},
        regions=[Region(Block([bound, split]))],
    )
    module = ModuleOp([plan])
    module.verify()
    text = dense_schedule_text(module)
    reparsed = parse_dense_schedule_plan(text)
    reparsed.verify()
    assert dense_schedule_text(reparsed) == text
    return text


print(split_decision())
try:
    split_decision(1)
except VerifyException as error:
    print("REFUSED:", error)
else:
    raise AssertionError("invalid split factor was accepted")


## 2. Check an instruction schedule

The **mk** representation in `merlin.sched.ir` describes counted loops and instruction calls. It is a different IR from the xDSL `schedule` dialect.

Here a small byte-copy kernel demonstrates the instruction checker. A backend normally supplies instruction meanings, byte footprints and encoding; this cell supplies a teaching declaration. Pointer offsets are in **bytes**. Extending the loop from eight to nine iterations must expose an out-of-bounds access. No target code is compiled.


In [ ]:
copy_interface = InstructionSet(
    target="illustrative-copy",
    instrs={
        "copy": InstrDef(
            "copy",
            (Operand("src", "ptr"), Operand("dst", "ptr")),
            render_c=lambda operands: f"*({operands[1]}) = *({operands[0]});",
            footprint=lambda values: [("src", 1, "read"), ("dst", 1, "write")],
        )
    },
    c_types={"i8": "int8_t"},
    drain_c="/* illustrative synchronous interface */",
)


def byte_copy(extent=8):
    return Kernel(
        "copy_bytes",
        (
            TensorArg("source", (8,), "i8", "read"),
            TensorArg("destination", (8,), "i8", "write"),
        ),
        (loop("i", extent, call("copy", src=Ptr("source", Var("i")), dst=Ptr("destination", Var("i")))),),
    )


kernel = byte_copy()
assert parse_kernel(kernel.text()).text() == kernel.text()
assert check_kernel(kernel, copy_interface) == []
violations = check_kernel(byte_copy(9), copy_interface)
assert violations
print(kernel.text())
print("Canonical text SHA256:", kernel.digest())
print("REFUSED extra iteration:", "\n".join(violations))


## 3. Schedule the matrix multiplication

Now use the shared `compiler.schedule.blocks` API. Its builder assigns operand rows, chooses command order and checks that loads and accumulators remain valid. This is a separate experiment, not a lowering of the split request above.

Try retaining loads, changing loop order or moving loads earlier with lookahead. The block edge represents the primitive geometry of our teaching machine; changing it models a different machine.

The picture shows assigned storage and **command order**, not elapsed time. Fewer loads do not by themselves establish a speedup.


In [ ]:
lhs = (np.arange(48).reshape(6, 8) % 7 - 3).astype(np.int8)
weight = (np.arange(48).reshape(8, 6) % 5 - 2).astype(np.int8)
reference = lhs.astype(np.int64) @ weight.astype(np.int64)
problem = Contraction(m=6, k=8, n=6)


def build_schedule(block=2, rows=64, retain=True, lookahead=1, order="knm", placement=OPPOSITE_END):
    geometry = Geometry(block, rows, 16, 64, True)
    knobs = Knobs(load_on_index_change=retain, lookahead_steps=lookahead, loop_order=tuple(order), placement=placement)
    result = schedule_contraction(problem, geometry, knobs)
    check_residency(result)
    np.testing.assert_array_equal(execute(result, lhs, weight), reference)
    return result


baseline = build_schedule()
without_retention = build_schedule(retain=False)
assert baseline.count(Load) < without_retention.count(Load)
print("Retained loads:", baseline.count(Load), "| reload in each group:", without_retention.count(Load))
print("Both schedules match the int64 fixture reference.")


In [ ]:
COLORS = {"Load": "#2563eb", "Preload": "#9333ea", "Compute": "#d97706", "Store": "#15803d"}


def schedule_view(schedule):
    # Every rectangle comes from the real scheduled operation or assigned region.
    counts = Counter(type(op).__name__ for op in schedule.ops)
    requested = sum(op.rows * op.cols for op in schedule.ops if isinstance(op, Load))
    labels = " | ".join(f"{kind}: {counts[kind]}" for kind in COLORS)
    width, left, span = 900, 110, 760
    svg = [
        f'<svg viewBox="0 0 {width} 195" role="img" aria-label="Assigned operand rows and ordered commands">',
        '<rect width="900" height="195" fill="white"/>',
    ]
    for lane, role in enumerate(("lhs", "weight")):
        base, rows = schedule.regions[role]
        y = 25 + lane * 45
        svg += [
            f'<text x="8" y="{y + 17}" fill="#111827">{role} rows</text>',
            f'<rect x="{left}" y="{y}" width="{span}" height="25" fill="#e5e7eb"/>',
            f'<rect x="{left + span * base / schedule.geometry.operand_rows}" y="{y}" '
            f'width="{span * rows / schedule.geometry.operand_rows}" height="25" fill="#bfdbfe"/>',
            f'<text x="{left + 4}" y="{y + 17}" fill="#111827">[{base}, {base + rows}) '
            f"of {schedule.geometry.operand_rows}</text>",
        ]
    svg += ['<text x="8" y="137" fill="#111827">Commands</text>']
    for index, op in enumerate(schedule.ops):
        color = COLORS[type(op).__name__]
        svg += [
            f'<rect x="{left + span * index / len(schedule.ops):.2f}" y="120" '
            f'width="{span / len(schedule.ops):.2f}" height="24" fill="{color}">'
            f"<title>{index}: {escape(str(op))}</title></rect>"
        ]
    svg += [
        f'<text x="{left}" y="168" fill="#111827">0 -> {len(schedule.ops) - 1} '
        "(command ordinal; hover for operands)</text>",
        "</svg>",
    ]
    markup = (
        "".join(svg) + f"<p>{escape(labels)}<br>Requested operand bytes: {requested} "
        "(i8 inputs; includes repeated loads). Residency: checked. Fixture result: exact.</p>"
    )
    show_html(markup, f"{schedule.regions}\n{labels}\nRequested operand bytes: {requested}")
    if schedule.notes:
        print("Placement notes:", *schedule.notes, sep="\n")


schedule_view(baseline)


## 4. See why schedules are refused

A small store can overwrite a still-live operand. Unknown bank geometry cannot justify separating operands by bank. A store before its reduction finishes reads an incomplete result.

Each cell below asks Merlin's existing builder or checker to detect the problem. The notebook implements no extra legality checker.


In [ ]:
def show_refusal(label, attempt):
    try:
        attempt()
    except (BlockScheduleError, ValueError) as error:
        print(f"{label}: REFUSED - {error}")
    else:
        raise AssertionError(f"{label}: expected a refusal")


show_refusal("Overlapping live rows", lambda: build_schedule(rows=32))
show_refusal(
    "Unknown bank geometry",
    lambda: schedule_contraction(problem, Geometry(2, 64, None, 64, True), Knobs(placement=BANK_ALIGNED)),
)
assert isinstance(baseline.ops[-1], Store)
show_refusal(
    "Store before reduction", lambda: check_residency(replace(baseline, ops=(baseline.ops[-1], *baseline.ops[:-1])))
)


## 5. Change the inputs

The controls call the same `build_schedule` function used above. Every accepted choice is checked against the integer matrix reference. Some combinations are expected to be refused.

Try turning retention off, setting store rows to 32, then changing block edge from 2 to 4. Compare loop order and lookahead using the command ribbon. Without widgets, edit a call such as `explore(block=4, rows=32, retain=False)`.


In [ ]:
def explore(block=2, rows=64, retain=True, lookahead=1, order="knm", placement=OPPOSITE_END):
    try:
        result = build_schedule(block, rows, retain, lookahead, order, placement)
    except BlockScheduleError as error:
        print("REFUSED:", error)
        return
    schedule_view(result)


explore()
try:
    import ipywidgets as widgets
except ImportError:
    print("Optional controls unavailable; call explore(block=4, rows=32, retain=False).")
else:
    controls = {
        "block": widgets.Dropdown(options=[2, 4], value=2, description="Block edge"),
        "rows": widgets.IntSlider(min=16, max=96, step=16, value=64, description="Store rows", continuous_update=False),
        "retain": widgets.Checkbox(value=True, description="Retain loads"),
        "lookahead": widgets.Dropdown(
            options=[("None ahead", 0), ("One step", 1), ("All ahead", None)], value=1, description="Lookahead"
        ),
        "order": widgets.Dropdown(options=["knm", "mnk", "nmk"], value="knm", description="Loop order"),
        "placement": widgets.Dropdown(
            options=[OPPOSITE_END, BANK_ALIGNED, CONTIGUOUS], value=OPPOSITE_END, description="Placement"
        ),
    }
    output = widgets.interactive_output(explore, controls)
    display(
        widgets.VBox(
            [
                widgets.HBox([controls["block"], controls["rows"], controls["retain"]]),
                widgets.HBox([controls["lookahead"], controls["order"], controls["placement"]]),
                output,
            ]
        )
    )


## Where to continue

| Question | Source owner |
|---|---|
| Which decisions exist? | `src/merlin/xdsl_dialects/schedule.py` |
| Which dense decisions are consumed? | `src/merlin/compiler/schedule/schedule_ir.py` |
| How are block commands built and checked? | `compiler/schedule/block.py` and `residency.py` |
| Where are other reusable passes? | `compiler/schedule/api_catalog.py` and the [shared-pass guide](../../docs/guides/shared_pass_compiler_generation.md) |
| Where is mk instruction IR defined? | `src/merlin/sched/ir/kernel.py` |

An **out-of-tree (OOT) backend** is a separate compiler package or repository. It supplies resource facts, numerical meaning, command effects and encoding. Its output can be LLVM IR, objects, device binaries or command artifacts.

This example checks a synchronous schedule and a small integer fixture. Async completion, quantization, physical encoding and actual device use need their own verification. Native MLIR/xDSL passes and the Transform dialect also keep their own invocation and representation; they are not interchangeable with these decisions or mk kernels.


**Step 3 of 5: Scheduling** | [All tutorials](README.md) | [Previous: Test generation](02_test_generation.ipynb) | [Next: Transformations](04_transformations.ipynb)
